In [1]:
import os, sys, json
import pandas as pd

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
from src.ml.config import load_ml_config
from src.ml.training_dataset import load_training_dataset
from src.ml.train import train_model, load_model, predict, retrain, list_runs

spark = get_spark_session("ml-train")
cfg = load_ml_config()

26/10/01 21:25:57 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/10/01 21:25:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 21:25:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Time to run

In [20]:
summary = train_model(spark, cfg)
print("run:", summary["run_id"])
pd.Series(summary["timings"], name="seconds").to_frame()

run: 20261001T171725341Z


,seconds
load_train_seconds,0.55
feature_engineering_seconds,2.18
model_fit_seconds,14.78
evaluation_seconds,3.19
profiling_seconds,0.24
save_model_seconds,1.89
total_seconds,23.33


### Sample predictions

In [12]:
model = load_model(cfg)
test = load_training_dataset(spark, cfg, split="test")
zones = test.select("pickup_location_id", "pickup_zone").distinct().toPandas()

p = predict(model, test, cfg).select("pickup_location_id", "hour_ts", "label", "prediction").toPandas()
p = p.merge(zones, on="pickup_location_id").sort_values(["pickup_location_id", "hour_ts"])
p["prediction"] = p["prediction"].round(1).clip(lower=0)
p["error"] = (p["prediction"] - p["label"]).round(1)
p[p["pickup_location_id"] == 161][["pickup_zone", "hour_ts", "label", "prediction", "error"]].head(24)

,pickup_zone,hour_ts,label,prediction,error
27570,Midtown Center,2024-03-18 12:00:00,134.0,133.7,-0.3
42,Midtown Center,2024-03-18 13:00:00,138.0,168.0,30.0
27792,Midtown Center,2024-03-18 14:00:00,197.0,205.6,8.6
28014,Midtown Center,2024-03-18 15:00:00,247.0,237.2,-9.8
28236,Midtown Center,2024-03-18 16:00:00,262.0,261.5,-0.5
264,Midtown Center,2024-03-18 17:00:00,299.0,270.5,-28.5
28458,Midtown Center,2024-03-18 18:00:00,318.0,291.7,-26.3
486,Midtown Center,2024-03-18 19:00:00,351.0,310.5,-40.5
28680,Midtown Center,2024-03-18 20:00:00,440.0,295.4,-144.6
28902,Midtown Center,2024-03-18 21:00:00,508.0,422.3,-85.7


### Root Mean Squared Error, Mean Absolute Error, R-squared, Weighted Absolute Percentage Error for model predictions of train, validation and test
### Aditionally some error calculations for based on 1hr, 1 day and 1 week demand for validation and test data

In [13]:
# model quality vs. naive baselines
rows = []
for split, entries in summary["metrics"].items():
    for name, m in entries.items():
        rows.append({"split": split, "predictor": name, **m})
pd.DataFrame(rows)

,split,predictor,rmse,mae,r2,wape,rows
0,train,model,9.8747,3.4493,0.9672,0.1818,301920
1,validation,model,14.6457,4.7756,0.9435,0.2155,72594
2,validation,baseline_demand_lag_1h,17.8652,5.8696,0.9159,0.2649,72594
3,validation,baseline_demand_lag_24h,25.3968,7.3492,0.8301,0.3317,72594
4,validation,baseline_demand_lag_168h,18.0319,5.7031,0.9144,0.2574,72594
5,test,model,13.8101,4.4998,0.9455,0.2084,72816
6,test,baseline_demand_lag_1h,17.5094,5.8361,0.9125,0.2702,72816
7,test,baseline_demand_lag_24h,22.2696,6.5347,0.8584,0.3026,72816
8,test,baseline_demand_lag_168h,14.0208,4.5453,0.9439,0.2105,72816


### Root Mean Squared Error, Mean Absolute Error, R-squared, Weighted Absolute Percentage Error for model predictions of train, validation and test with accuracy

In [14]:
import pandas as pd
rows = []
for split in ("train", "validation", "test"):
    m = summary["metrics"][split]["model"]
    rows.append({"split": split, "RMSE": m["rmse"], "MAE": m["mae"], "R2": m["r2"], "WAPE": m["wape"],
                 "accuracy_%": round((1 - m["wape"]) * 100, 1)})
pd.DataFrame(rows)

,split,RMSE,MAE,R2,WAPE,accuracy_%
0,train,9.8747,3.4493,0.9672,0.1818,81.8
1,validation,14.6457,4.7756,0.9435,0.2155,78.5
2,test,13.8101,4.4998,0.9455,0.2084,79.2


### Stage info

In [4]:
# per-stage cost: stages, tasks, shuffle, spill, I/O
rows = [{"stage": s, **{k: v for k, v in r.items() if k != "spark"}, **(r.get("spark") or {})}
        for s, r in summary["stages"].items()]
pd.DataFrame(rows)

,stage,seconds,peak_memory_mb,stages,executor_run_seconds,tasks,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
0,load_train,1.91,629.3,4,0.73,4,21142108,0,59,59,0,0
1,feature_engineering,5.40,2415.8,15,20.82,53,183007800,0,47032848,47032848,0,0
2,model_fit,17.22,3317.2,604,83.29,4825,136633339936,0,180250425,180250425,0,0
3,evaluation,4.06,3797.3,91,10.47,227,855527347,0,30397919,30397919,0,0
4,profiling,0.25,3808.4,4,0.18,4,38776272,0,150,150,0,0
5,save_model,1.38,3829.9,22,0.61,55,0,339130,452464,452464,0,0


### Processing effort

In [5]:
# preprocessing effort: how much missing data had to be handled, and what was left out
print(json.dumps(summary["preprocessing"], indent=2))

{
  "feature_pipeline_stages": 8,
  "numeric_features": 19,
  "categorical_features": 3,
  "feature_vector_size": 263,
  "imputed_columns": [
    "weather_precipitation",
    "air_quality_pm25"
  ],
  "null_share_before_imputation": {
    "weather_temperature": 0.0,
    "weather_relative_humidity": 0.0,
    "weather_precipitation": 0.0809,
    "weather_wind_speed": 0.0,
    "weather_pressure": 0.0,
    "air_quality_pm25": 0.3035,
    "demand_lag_1h": 0.0,
    "demand_lag_24h": 0.0,
    "demand_lag_168h": 0.0,
    "demand_rolling_mean_24h": 0.0
  },
  "columns_not_used": [
    "air_quality_station_count",
    "pickup_zone"
  ]
}


### Reproducibility check

In [15]:
# reproducibility fingerprints
print(json.dumps(summary["reproducibility"], indent=2, default=str))
pd.Series(summary["importance_by_dataset"], name="importance share").to_frame()

{
  "seed": 42,
  "config_hash": "7152862e3de6d8c0",
  "code_version": "b8b86c2eea5544beb095888c3236fa7d33dd37a5",
  "train_data_checksum": 7643688161793619368,
  "predictions_checksum": -181127304390398091,
  "training_dataset": {
    "source_gold_version": 1,
    "training_dataset_version": 1,
    "time_range_utc": {
      "start": "2024-01-01 05:00:00",
      "end": "2024-04-01 03:00:00"
    },
    "split_boundaries_utc": {
      "train_end": "2024-03-04 21:00:00",
      "validation_end": "2024-03-18 12:00:00"
    },
    "splits": {
      "train": {
        "rows": 301920,
        "zones": 222,
        "mean_target": 18.9753,
        "first_hour": "2024-01-08 05:00:00",
        "last_hour": "2024-03-04 20:00:00"
      },
      "validation": {
        "rows": 72594,
        "zones": 222,
        "mean_target": 22.1587,
        "first_hour": "2024-03-04 21:00:00",
        "last_hour": "2024-03-18 11:00:00"
      },
      "test": {
        "rows": 72816,
        "zones": 222,
        "

,importance share
tripdata (demand history),0.7028
calendar (from trip timestamps),0.1748
taxi_zone_lookup,0.0656
weather,0.0429
air_quality,0.0139


### Reproducibility prove with same checksum

In [21]:
# Train twice on the same data: the checksums should match.
list_runs(cfg)

,run_id,algorithm,training_rows,load_train_seconds,feature_engineering_seconds,model_fit_seconds,evaluation_seconds,profiling_seconds,save_model_seconds,total_seconds,validation_rmse,test_rmse,test_wape,train_data_checksum,predictions_checksum
0,20261001T160327296Z,gbt,301920,1.91,5.40,17.22,4.06,0.25,1.38,31.11,14.6457,13.8101,0.2084,7643688161793619368,-181127304390398091
1,20261001T171725341Z,gbt,301920,0.55,2.18,14.78,3.19,0.24,1.89,23.33,14.6457,13.8101,0.2084,7643688161793619368,-181127304390398091


In [18]:
print(spark.sparkContext.uiWebUrl) 

http://192.168.1.8:4042


### Run with new data

In [10]:
#  1. April data is in Gold (Week 3 incremental ingestion + Gold incremental build done)
#  2. data_end in config/ml.yaml moved forward to cover April

import shutil, os
from src.ml.config import load_ml_config
from src.ml.train import load_model, retrain, list_runs

cfg = load_ml_config("config/ml.yaml") 

src_dir = cfg.model.directory
backup_dir = src_dir + "_v1"
if os.path.exists(src_dir) and not os.path.exists(backup_dir):
    shutil.copytree(src_dir, backup_dir)

# rebuild the training dataset from Gold (with April) and train again with the same code
summary2 = retrain(spark, cfg, refresh_dataset=True)
print("run_id:", summary2["run_id"])
print("test metrics:", summary2["metrics"]["test"]["model"])

list_runs(cfg)

run_id: 20261001T193022907Z
test metrics: {'rmse': 13.9122, 'mae': 3.901, 'r2': 0.518, 'wape': 0.6772, 'rows': 83916}


,run_id,algorithm,training_rows,load_train_seconds,feature_engineering_seconds,model_fit_seconds,evaluation_seconds,profiling_seconds,save_model_seconds,total_seconds,validation_rmse,test_rmse,test_wape,train_data_checksum,predictions_checksum
0,20261001T160327296Z,gbt,301920,1.91,5.40,17.22,4.06,0.25,1.38,31.11,14.6457,13.8101,0.2084,7643688161793619368,-181127304390398091
1,20261001T171725341Z,gbt,301920,0.55,2.18,14.78,3.19,0.24,1.89,23.33,14.6457,13.8101,0.2084,7643688161793619368,-181127304390398091
2,20261001T192642899Z,gbt,354312,0.61,3.08,16.94,4.36,0.23,0.98,27.03,13.7539,13.9122,0.6772,-3231943143820098138,-3181539304827170135
3,20261001T193022907Z,gbt,354312,0.33,2.86,16.22,3.45,0.21,0.90,24.40,13.7539,13.9122,0.6772,-3231943143820098138,-3181539304827170135


In [ ]:
spark.stop()